# Component-level causal attribution (Experiment 9)
Only needed if the section keeps the title *Mechanistic account*. We decompose the competitor margin (z_b - z_a) by layer/component via direct logit attribution, then ablate the top late attention components and measure the effect on the margin and the gold rank. If you retitle to *Selection-margin account*, this can live in an appendix.

In [ ]:
# --- environment (pins matching the pipeline) ---
# transformers==4.46.2  numpy==1.26.4  ; PyTorch nightly cu128 on newer instances.
import os, gc, json, math, pathlib
import numpy as np, torch
from tqdm.auto import tqdm
import rw_core as rc          # tested core (rw_core_smoketest.py: 19/19)
import rw_modelio as mio      # model IO / hooks / generation

ART = pathlib.Path(os.environ.get("RW_ART", "artifacts")); ART.mkdir(exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def save(obj, name):
    p = ART / name
    np.savez_compressed(p, **obj) if name.endswith(".npz") else \
        p.write_text(json.dumps(obj, indent=2, default=float))
    print("saved", p)

def exists(name):  # skip-if-exists guard
    return (ART / name).exists()


In [ ]:
MODELS = [
    "meta-llama/Llama-3.1-8B", "meta-llama/Llama-3.2-3B", "meta-llama/Llama-3.2-1B",
    "meta-llama/Llama-3.2-3B-Instruct",
    "Qwen/Qwen2.5-3B", "Qwen/Qwen2.5-3B-Instruct", "Qwen/Qwen2.5-7B",
    "mistralai/Mistral-7B-v0.1",
]

In [ ]:
# ============================================================================
# DATA ADAPTER  --  wire this to your existing read/write cache.
# Return a list of item dicts (schema in rw_core docstring). The fields used
# downstream are listed per-cell. This is the ONLY place that knows your layout.
# ============================================================================
def load_items(model_name):
    """TODO: load your per-item records for `model_name`.
    Required keys vary by notebook; each cell asserts what it needs."""
    raise NotImplementedError("wire load_items() to your read/write cache")


In [ ]:
# DLA on the margin per item, aggregated over readable-unselected failures.
from transformers import AutoModelForCausalLM, AutoTokenizer
def run_dla(name, max_items=300):
    model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.bfloat16,
                                                 device_map=DEVICE).eval()
    tok = AutoTokenizer.from_pretrained(name)
    W, final_norm, _ = mio.get_unembedding(model)
    W_gpu = torch.tensor(W.numpy(), device=DEVICE)
    nL = model.config.num_hidden_layers
    late = mio.band_indices(nL, 0.6, 1.0)          # late layers for per-head detail
    items = [it for it in load_items(name)
             if it.get("readable_unselected", not it["is_direct_success"])][:max_items]
    agg = {}
    for it in tqdm(items):
        writes = mio.capture_component_writes(model, tok, it["prompt"], late, DEVICE)
        # frozen-stat linearization of the final RMSNorm at this item's h^L
        cap = mio.capture_hidden_states(model, tok, it["prompt"], DEVICE)
        ln_scale = rc.ln_scale_from_state(cap["hs"][-1],
                    weight=final_norm.weight.detach().float().cpu().numpy())
        contrib = rc.dla_margin_contributions(writes, W.numpy(),
                    it["gold_first_tok"], it["alt_id"], ln_scale)
        for k,v in contrib.items(): agg[k] = agg.get(k,0.0)+v/len(items)
    del model; gc.collect(); torch.cuda.empty_cache()
    return dict(sorted(agg.items(), key=lambda kv: -kv[1]))

dla = {}
for name in MODELS:
    if exists(f"dla_{name.split('/')[-1]}.json"): continue
    dla[name] = run_dla(name)
    save(dla[name], f"dla_{name.split('/')[-1]}.json")
    print(name, "top margin contributors:",
          list(dla[name].items())[:6])

Positive contributors push toward the competitor (against the gold). The expected result, matching your trajectory finding, is that a small set of **late attention** components dominate the positive side of the margin. Report per model which components they are.

In [ ]:
# Ablation: zero the top-k late attention components, recompute rank/margin.
@torch.no_grad()
def ablate_topk(name, k=3, max_items=200):
    model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.bfloat16,
                                                 device_map=DEVICE).eval()
    tok = AutoTokenizer.from_pretrained(name)
    top = [c for c in json.loads((ART/f"dla_{name.split('/')[-1]}.json").read_text())
           if ".attn" in c][:k]
    # parse "L{l}.attn(.h{h})" -> register hooks that zero those writes
    handles = []
    for comp in top:
        l = int(comp.split(".")[0][1:])
        layer = model.model.layers[l-1]
        if ".h" in comp:
            head = int(comp.split(".h")[1]); hd = model.config.hidden_size//model.config.num_attention_heads
            def mk(head=head, hd=hd):
                def hook(mod, inp, out):
                    x = list(inp); v = x[0].clone(); v[..., head*hd:(head+1)*hd] = 0
                    return mod._orig_forward(v) if hasattr(mod,"_orig_forward") else out
                return hook
            handles.append(layer.self_attn.o_proj.register_forward_pre_hook(
                lambda m,i,head=head,hd=hd: (i[0].index_fill(-1,
                    torch.arange(head*hd,(head+1)*hd,device=i[0].device),0.0),)))
        else:
            def hook(mod, inp, out):
                o = out[0] if isinstance(out,tuple) else out; o.zero_(); return out
            handles.append(layer.self_attn.register_forward_hook(hook))
    items = [it for it in load_items(name)
             if it.get("readable_unselected", not it["is_direct_success"])][:max_items]
    flipped = []
    for it in items:
        ids = tok(it["prompt"], return_tensors="pt").to(DEVICE)
        z = model(**ids).logits[0,-1]
        flipped.append(int(z.argmax())==it["gold_first_tok"])
    for h in handles: h.remove()
    del model; gc.collect(); torch.cuda.empty_cache()
    return float(np.mean(flipped))

ab = {n: ablate_topk(n) for n in MODELS if exists(f"dla_{n.split('/')[-1]}.json")}
save(ab, "exp9_ablation_recovery.json"); print(ab)

If zeroing the top late-attention components flips a meaningful fraction of failures to the gold, the trajectory result becomes causal and the word *mechanistic* is earned. Otherwise retitle to *Selection-margin account / Final-readout geometry* and move DLA to an appendix.